In [0]:
%run ./Includes/Copy-Datasets

In [0]:
files = dbutils.fs.ls(f"{dataset_bookstore}/kafka-raw")
display(files)

In [0]:
df_raw = spark.read.json(f"{dataset_bookstore}/kafka-raw")
display(df_raw)

In [0]:
from pyspark.sql import functions as functions

def process_bronze():
    schema = "key BINARY, offset long, partition long, timestamp long, topic string, value BINARY"
    query = (spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format", "json")
                    .schema(schema)
                    .load(f"{dataset_bookstore}/kafka-raw")
                    .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))
                    .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
                .writeStream
                    .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/bronze")
                    .option("mergeSchema", "true")
                    .partitionBy("topic", "year_month")
                    .trigger(availableNow=True)
                    .table("bronze"))
    
    query.awaitTermination()
                    

In [0]:
process_bronze()

In [0]:
batch_df = spark.table("bronze")
display(batch_df)

In [0]:
%sql
SELECT DISTINCT topic FROM bronze;

In [0]:
bookstore.load_new_data()

In [0]:
process_bronze()

In [0]:
%sql
SELECT COUNT(*) FROM bronze;